In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load merged PKL files (assumes you saved after pd.merge on TransactionID)
df_train = pd.read_pickle('merged_train.pkl')
df_test = pd.read_pickle('merged_test.pkl')

# Drop target if present (prevents leakage)
if 'isFraud' in df_train.columns:
    y_train = df_train['isFraud'].copy()  # Save separately for modeling
    df_train = df_train.drop('isFraud', axis=1)
else:
    y_train = None

print(f"Loaded train shape: {df_train.shape}, test shape: {df_test.shape}")

# Memory optimization function
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
            elif c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                df[col] = df[col].astype(np.float16)
    return df

df_train = reduce_mem_usage(df_train)
df_test = reduce_mem_usage(df_test)

# Time features from TransactionDT
for df in [df_train, df_test]:
    df['Transaction_hour'] = (df['TransactionDT'] / 3600) % 24
    df['Transaction_day'] = df['TransactionDT'] // (24 * 3600)
    df['Transaction_weekend'] = ((df['Transaction_hour'] >= 12) & (df['Transaction_hour'] <= 18)).astype(int)

# Amount transformations
for df in [df_train, df_test]:
    df['Transaction_amt_log'] = np.log1p(df['TransactionAmt'].fillna(0))
    df['amt_decimal'] = ((df['TransactionAmt'] % 1).fillna(0) * 1000).astype(np.int16)

# UID features (key for fraud patterns)
for df in [df_train, df_test]:
    df['card_id'] = df['card1'].astype(str) + '_' + df['card2'].fillna('nan').astype(str) + '_' + df['card3'].fillna('nan').astype(str) + '_' + df['card5'].fillna('nan').astype(str)
    df['addr_id'] = df['addr1'].fillna(-999).astype(int).astype(str) + '_' + df['addr2'].fillna(-999).astype(int).astype(str)
    df['uid'] = df['card_id'] + '_' + df['addr_id']

# Label encode categoricals (fit on combined to handle new test cats)
cats = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'DeviceType', 'DeviceInfo']
for col in cats:
    if col in df_train.columns:
        le = LabelEncoder()
        combined = pd.concat([df_train[col], df_test[col]]).astype(str).fillna('missing')
        le.fit(combined)
        df_train[col] = le.transform(df_train[col].astype(str).fillna('missing'))
        df_test[col] = le.transform(df_test[col].astype(str).fillna('missing'))

# Fill NAs uniformly
df_train.fillna(-999, inplace=True)
df_test.fillna(-999, inplace=True)

# Aggregations (group stats per UID/card - train first, then test)
uid_aggs = df_train.groupby('uid')['Transaction_amt_log'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
uid_aggs.columns = ['uid', 'uid_cnt', 'uid_amt_mean', 'uid_amt_std', 'uid_amt_min', 'uid_amt_max']
df_train = df_train.merge(uid_aggs, on='uid', how='left')

uid_aggs_test = df_test.groupby('uid')['Transaction_amt_log'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
uid_aggs_test.columns = ['uid', 'uid_cnt', 'uid_amt_mean', 'uid_amt_std', 'uid_amt_min', 'uid_amt_max']
df_test = df_test.merge(uid_aggs_test, on='uid', how='left').fillna(-999)

# Drop high-missing/UID cols
drop_cols = [col for col in df_train.columns if 'id_' in col and df_train[col].nunique() < 10] + ['uid', 'card_id', 'addr_id']
df_train.drop(drop_cols, axis=1, inplace=True, errors='ignore')
df_test.drop(drop_cols, axis=1, inplace=True, errors='ignore')

# Final shapes and save engineered PKLs
print(f"Engineered train shape: {df_train.shape}")
print(f"Engineered test shape: {df_test.shape}")

df_train.to_pickle('engineered_train.pkl')
df_test.to_pickle('engineered_test.pkl')
if y_train is not None:
    np.save('y_train.npy', y_train.values)  # For ADASYN/XGBoost

print("Ready for XGBoost + ADASYN!")


Loaded train shape: (590540, 433), test shape: (506691, 433)
Engineered train shape: (590540, 430)
Engineered test shape: (506691, 430)
Ready for XGBoost + ADASYN!
